<a href="https://colab.research.google.com/github/SinaRampe/Health-Topic-Classifier/blob/main/train_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Train a classifier that decides, whether a question is about a health care topic.

Install and import dependencies.

In [ ]:
! pip install transformers
! pip install umap-learn==0.5.1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import torch
import transformers
import pickle
from umap import UMAP
from sklearn.preprocessing import MinMaxScaler

In [ ]:
!git clone https://github.com/SinaRampe/Health-Topic-Classifier.git

In [ ]:
!ls Health-Topic-Classifier/data

### Load datasets.

In [ ]:
train = pd.read_csv("Health-Topic-Classifier/data/classifier_train.csv")
test = pd.read_csv("Health-Topic-Classifier/data/classifier_test.csv")
full = pd.read_csv("Health-Topic-Classifier/data/df_full.csv")

### Tokenizer setup.

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["prompt"], padding=True, truncation=True)

In [ ]:
train_encoded = train.map(tokenize, batched=True, batch_size=None)
test_encoded = test.map(tokenize, batched=True, batch_size=None)
full_encoded = full.map(tokenize, batched=True, batch_size=None)

## Training a Text Classifier.

#### Extract hidden states.

In [ ]:
from transformers import AutoModel

model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_ckpt).to(device)

In [ ]:
def extract_hidden_states(batch):
    # Place model inputs on the GPU
    inputs = {k:v.to(device) for k,v in batch.items() 
              if k in tokenizer.model_input_names}
    # Extract last hidden states
    with torch.no_grad():
        last_hidden_state = model(**inputs).last_hidden_state
    # Return vector for [CLS] token
    return {"hidden_state": last_hidden_state[:,0].cpu().numpy()}

In [ ]:
train_encoded.set_format("torch", 
                            columns=["input_ids", "attention_mask", "completion"])
test_encoded.set_format("torch", 
                            columns=["input_ids", "attention_mask", "completion"])
full_encoded.set_format("torch", 
                            columns=["input_ids", "attention_mask", "completion"])

In [ ]:
train_hidden = train_encoded.map(extract_hidden_states, batched=True)
test_hidden = test_encoded.map(extract_hidden_states, batched=True)
full_hidden = full_encoded.map(extract_hidden_states, batched=True)

In [ ]:
train_hidden["train"].column_names

#### Extract features and visualize the training set.

In [ ]:
X_train = np.array(train_hidden["train"]["hidden_state"])
X_valid = np.array(test_hidden["train"]["hidden_state"])
y_train = np.array(train_hidden["train"]["completion"])
y_valid = np.array(test_hidden["train"]["completion"])
X_full = np.array(full_hidden["train"]["hidden_state"])
y_full = np.array(full_hidden["train"]["completion"])
X_train.shape, X_valid.shape, X_full.shape

For visualizing the hidden states (768 D) we project the vectors down to 2D with UMAP after applying MinMaxScaler.

In [ ]:
scaler = MinMaxScaler()
umap = UMAP(n_components=2, metric="cosine")
# Scale features to [0,1] range
X_scaled = scaler.fit_transform(X_train)
# Initialize and fit UMAP
mapper = umap.fit(X_scaled)
# Create a DataFrame of 2D embeddings
df_emb = pd.DataFrame(mapper.embedding_, columns=["X", "Y"])
df_emb["label"] = y_train
df_emb.head()

Plot the density. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,5))
axes = axes.flatten()
cmaps = ["Blues", "Reds"]
labels = ["not health-related questions", "health-related questions"]

for i, (label, cmap) in enumerate(zip(labels, cmaps)):
    df_emb_sub = df_emb.query(f"label == {i}")
    axes[i].hexbin(df_emb_sub["X"], df_emb_sub["Y"], cmap=cmap,
                   gridsize=20, linewidths=(0,))
    axes[i].set_title(label)
    axes[i].set_xticks([]), axes[i].set_yticks([])

plt.tight_layout()
plt.show()

#### Train a simple classifier.


Use the hidden states to train a logistic regression model.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=5000)
#lr_clf.fit(X_train, y_train)
lr_clf.fit(X_full, y_full)

In [ ]:
#lr_clf.score(X_valid, y_valid) -> was 0.98 on test, which is sufficient for our task

Lets look at one example:

In [ ]:
print("Question: ", full_hidden["train"]["prompt"][32])
print("Prediction: ", lr_clf.predict(np.array(full_hidden["train"]["hidden_state"][32]).reshape(1, -1))[0])

#### Lets classify an invented question.

In [ ]:
text = "What is Ibuprofen?"
inputs = tokenizer(text, return_tensors="pt")
print(f"Input tensor shape: {inputs['input_ids'].size()}")

In [ ]:
print("Question: ", text)
print("Prediction: ",lr_clf.predict(extract_hidden_states(inputs)["hidden_state"])[0])

### Further investigations of the performance

####Confusion matrix.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def plot_confusion_matrix(y_preds, y_true, labels):
    cm = confusion_matrix(y_true, y_preds, normalize="true")
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()
    
y_preds = lr_clf.predict(X_valid)
plot_confusion_matrix(y_preds, y_valid, labels)

#### Let's take a look at the false predictions. -> Not too bad!

In [ ]:
classification_is_wrong = lr_clf.predict(X_valid) != y_valid
indizes = [i for i, x in enumerate(classification_is_wrong) if x]

In [ ]:
test["train"][indizes]

#### Dump the trained classifier.

In [ ]:
pickle.dump(lr_clf, open("log_regr_classifier.sav", 'wb'))